# 02 — Step 1: Selectivity Score Screening

**Goal**: For each self-attention head in SD1.5's UNet, compute a Reflection Selectivity Score:

$$S(h, l) = \text{CRA}_{\text{mirror}}(h, l) - \text{CRA}_{\text{nonmirror}}(h, l)$$

SD1.5 has 16 self-attention blocks × 8 heads = 128 head positions to screen.

In [ ]:
import sys
sys.path.insert(0, '/content/mi-mirror')

import torch
import json
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

from scripts.config import (
    MIRROR_PROMPTS, NON_MIRROR_PROMPTS, SEEDS,
    EXPERIMENTS_DIR, NUM_HEADS, NUM_INFERENCE_STEPS,
    TOP_K_CANDIDATES,
)
from scripts.model_loader import load_pipeline
from scripts.roi import get_default_rois
from scripts.generate import generate_with_cra
from scripts.attention_extraction import AttentionData, discover_self_attn_blocks
from scripts.metrics import compute_selectivity_matrix, rank_candidates
from scripts.visualization import plot_selectivity_heatmap, plot_top_candidates

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP1_DIR.mkdir(parents=True, exist_ok=True)

obj_roi, ref_roi = get_default_rois()
print(f"Object ROI: {obj_roi.name}, Reflection ROI: {ref_roi.name}")

In [ ]:
pipe = load_pipeline()

# Discover attention blocks
block_infos = discover_self_attn_blocks(pipe.unet)
n_blocks = len(block_infos)
print(f"Found {n_blocks} self-attention blocks:")
for b in block_infos:
    print(f"  [{b.linear_idx:2d}] {b.position} block {b.block_idx}, layer {b.layer_idx} — {b.resolution}x{b.resolution}")

In [ ]:
# Run CRA extraction on mirror prompts
print("=== Mirror Prompts ===")
mirror_attention_data = []

for i, prompt in enumerate(MIRROR_PROMPTS):
    for seed in SEEDS:
        print(f"  Prompt {i}, seed {seed}...", end=" ")
        attn_data = AttentionData()
        image, attn_data, _ = generate_with_cra(
            pipe, prompt, seed, obj_roi, ref_roi, attn_data
        )
        mirror_attention_data.append(attn_data)
        print(f"done ({len(attn_data.cra_scalars)} CRA values)")

print(f"\nCollected {len(mirror_attention_data)} mirror samples")

In [ ]:
# Run CRA extraction on non-mirror prompts
print("=== Non-Mirror Prompts ===")
nonmirror_attention_data = []

for i, prompt in enumerate(NON_MIRROR_PROMPTS):
    for seed in SEEDS:
        print(f"  Prompt {i}, seed {seed}...", end=" ")
        attn_data = AttentionData()
        image, attn_data, _ = generate_with_cra(
            pipe, prompt, seed, obj_roi, ref_roi, attn_data
        )
        nonmirror_attention_data.append(attn_data)
        print(f"done ({len(attn_data.cra_scalars)} CRA values)")

print(f"\nCollected {len(nonmirror_attention_data)} non-mirror samples")

In [ ]:
# Compute selectivity matrix
selectivity = compute_selectivity_matrix(
    mirror_attention_data, nonmirror_attention_data,
    n_blocks=n_blocks,
)
print(f"Selectivity matrix shape: {selectivity.shape}")
print(f"Max selectivity: {selectivity.max():.6f}")
print(f"Min selectivity: {selectivity.min():.6f}")
print(f"Mean selectivity: {selectivity.mean():.6f}")

In [ ]:
# Visualize selectivity heatmap
fig = plot_selectivity_heatmap(selectivity, block_infos=block_infos)
fig.savefig(STEP1_DIR / "selectivity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank and display top candidates
candidates = rank_candidates(selectivity, top_k=TOP_K_CANDIDATES)
info_map = {b.linear_idx: b for b in block_infos}

print(f"\nTop-{TOP_K_CANDIDATES} Candidate Reflection Heads:")
print(f"{'Rank':<6} {'Block':<20} {'Head':<8} {'Selectivity':<12}")
print("-" * 46)
for rank, (b, h, s) in enumerate(candidates, 1):
    info = info_map[b]
    label = f"{info.position} {info.block_idx}.{info.layer_idx} ({info.resolution}x{info.resolution})"
    print(f"{rank:<6} {label:<20} {h:<8} {s:<12.6f}")

In [ ]:
fig = plot_top_candidates(candidates, block_infos=block_infos)
fig.savefig(STEP1_DIR / "top_candidates.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save results
results = {
    "selectivity_matrix": selectivity,
    "candidates": candidates,
    "block_infos": block_infos,
    "n_blocks": n_blocks,
    "mirror_attention_data": mirror_attention_data,
    "nonmirror_attention_data": nonmirror_attention_data,
}
with open(STEP1_DIR / "step1_results.pkl", "wb") as f:
    pickle.dump(results, f)

candidates_json = [
    {"linear_idx": b, "head": h, "selectivity": s,
     "position": info_map[b].position, "block_idx": info_map[b].block_idx,
     "layer_idx": info_map[b].layer_idx, "resolution": info_map[b].resolution}
    for b, h, s in candidates
]
with open(STEP1_DIR / "candidates.json", "w") as f:
    json.dump(candidates_json, f, indent=2)

print(f"Results saved to {STEP1_DIR}")